# Fase 2 — Exploração das bases SIM e SINASC (2024)

Notebook de análise exploratória usado para embasar as decisões de tratamento do ETL. As decisões aqui registradas são implementadas nos scripts `06` em diante.

**Fontes:** microdados do SIM e do SINASC (OpenDataSUS) e tabela oficial da CID-10 (DATASUS).

## 1. Configuração

Importa as bibliotecas e define os caminhos dos arquivos brutos a partir da raiz do projeto. Os CSVs não são versionados no GitHub por causa do tamanho (~550 MB cada).

In [1]:
import pandas as pd
from pathlib import Path

RAIZ = Path.cwd().parents[1]
BRUTOS = RAIZ / "dados" / "brutos"

ARQUIVOS = {
    "SINASC": BRUTOS / "sinasc_2024.csv",
    "SIM": BRUTOS / "sim_2024.csv",
}

pd.set_option("display.max_columns", None)  # mostra todas as colunas
print(RAIZ)

c:\Users\Livia\Documents\Proj-Big-Data


## 2. Volume das bases

Conta o total de registros de cada arquivo. A leitura é feita em blocos (`chunksize`) e apenas da primeira coluna, para não carregar o arquivo inteiro na memória.
**Resultado:** SINASC 2024 com 2.389.325 registros (536 MB) e SIM 2024 com 1.532.015 registros (525 MB).

In [ ]:
for nome, caminho in ARQUIVOS.items():
    total = 0
    for bloco in pd.read_csv(caminho, sep=";", encoding="latin-1",
                             usecols=[0], dtype=str, chunksize=500_000):
        total += len(bloco)
    tamanho_mb = caminho.stat().st_size / 1024**2
    print(f"{nome}: {total:,} linhas | {tamanho_mb:,.0f} MB".replace(",", "."))

## 3. Amostra do SINASC

Visualização das 100 primeiras linhas para entender o formato de cada coluna. Todas as colunas são lidas como texto (`dtype=str`) para preservar zeros à esquerda.

**Observação:** o SINASC possui 62 colunas. Existem duas colunas de raça/cor: `RACACOR` (do recém-nascido) e `RACACORMAE` (da mãe). Como a análise é sobre as mães, será usada a `RACACORMAE`.

In [3]:
sinasc = pd.read_csv(ARQUIVOS["SINASC"], sep=";", encoding="latin-1",
                     nrows=100, dtype=str)
print(f"{sinasc.shape[1]} colunas")
sinasc

62 colunas


,contador,ORIGEM,CODESTAB,CODMUNNASC,LOCNASC,IDADEMAE,ESTCIVMAE,ESCMAE,CODOCUPMAE,QTDFILVIVO,QTDFILMORT,CODMUNRES,GESTACAO,GRAVIDEZ,PARTO,CONSULTAS,DTNASC,HORANASC,SEXO,APGAR1,APGAR5,RACACOR,PESO,IDANOMAL,DTCADASTRO,CODANOMAL,NUMEROLOTE,VERSAOSIST,DTRECEBIM,DIFDATA,OPORT_DN,DTRECORIGA,NATURALMAE,CODMUNNATU,CODUFNATU,ESCMAE2010,SERIESCMAE,DTNASCMAE,RACACORMAE,QTDGESTANT,QTDPARTNOR,QTDPARTCES,IDADEPAI,DTULTMENST,SEMAGESTAC,TPMETESTIM,CONSPRENAT,MESPRENAT,TPAPRESENT,STTRABPART,STCESPARTO,TPNASCASSI,TPFUNCRESP,TPDOCRESP,DTDECLARAC,ESCMAEAGR1,STDNEPIDEM,STDNNOVA,CODPAISRES,TPROBSON,PARIDADE,KOTELCHUCK
0,1,1,2516500,110001,1,24,1,3,999992,01,00,110001,5,1,2,2,14022024,0845,1,08,09,4,3120,2,05032024,NaN,20240003,3.2.50,05032024,020,20,05032024,811,110028,11,2,5,11012000,4,01,00,01,NaN,17052023,38,8,02,03,1,2,2,1,2,3,14022024,03,0,1,1,05,1,2
1,2,1,2516500,110001,1,29,2,5,999992,00,00,110001,5,1,2,4,17042024,0850,1,08,09,4,3564,2,09052024,NaN,20240005,3.2.50,09052024,022,22,09052024,811,110001,11,5,NaN,22111994,4,00,00,00,41,17072023,39,8,08,05,1,2,1,1,2,3,17042024,08,0,1,1,02,0,2
2,3,1,2516500,110001,1,30,5,4,622020,02,00,110001,5,1,2,4,29052024,1006,1,08,09,4,2816,2,03062024,NaN,20240006,3.2.50,03062024,005,5,03062024,811,110001,11,3,2,12081993,4,02,00,02,35,27082023,39,8,08,02,1,2,1,1,2,3,29052024,05,0,1,1,05,1,5
3,4,1,2516500,110001,1,14,5,4,999991,00,00,110001,5,1,2,4,27052024,0900,1,08,09,1,3126,2,03062024,NaN,20240006,3.2.50,03062024,007,7,03062024,811,110001,11,2,8,12102009,1,00,00,00,17,28082023,38,8,10,03,1,2,2,1,2,3,27052024,04,0,1,1,01,0,5
4,5,1,2516500,110001,1,24,2,4,999992,01,00,110001,5,1,2,4,13052024,0835,1,08,09,4,3622,2,03062024,NaN,20240006,3.2.50,03062024,021,21,03062024,850,500270,50,3,1,23091999,4,01,00,01,29,08082023,39,8,11,02,1,2,2,1,2,3,13052024,05,0,1,1,05,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,1,2494299,110002,1,29,1,4,512120,01,00,110140,4,1,1,3,17012024,0104,2,08,09,1,3075,2,17012024,NaN,20240004,3.2.50,24012024,007,7,24012024,811,110140,11,3,2,21081994,1,01,01,00,NaN,19052023,34,8,05,02,1,2,3,2,2,3,17012024,05,1,1,1,10,1,3
96,97,1,2494299,110002,1,43,1,4,763015,02,01,110002,4,1,2,4,16012024,1634,2,08,09,4,3280,2,17012024,NaN,20240004,3.2.50,24012024,008,8,24012024,835,350400,35,3,3,28111980,4,03,01,01,NaN,03052023,36,8,12,02,1,2,2,1,2,3,16012024,06,1,1,1,10,1,5
97,98,1,2494299,110002,1,23,1,4,999992,01,00,110026,5,1,2,4,16012024,1546,2,08,09,4,3560,2,17012024,NaN,20240004,3.2.50,24012024,008,8,24012024,811,110013,11,2,8,21062000,4,01,01,00,NaN,14042023,39,8,13,02,1,2,1,1,2,3,16012024,04,1,1,1,04,1,5
98,99,1,2515520,110002,1,33,1,4,524305,01,00,110002,5,1,2,4,18012024,0728,2,08,09,4,2930,2,18012024,NaN,20240004,3.2.50,24012024,006,6,24012024,811,110060,11,3,3,02061990,4,01,00,01,NaN,NaN,38,1,07,01,1,2,1,1,2,3,18012024,06,1,1,1,05,1,5


## 4. Amostra do SIM

Visualização das 100 primeiras linhas do SIM (88 colunas).

**Observação:** nenhuma das 100 primeiras linhas é óbito materno, pois eles representam uma fração muito pequena do total. A análise das colunas do SIM é feita sobre os casos filtrados (seção 6).

In [ ]:
sim = pd.read_csv(ARQUIVOS["SIM"], sep=";", encoding="latin-1",
                  nrows=100, dtype=str)
print(f"{sim.shape[1]} colunas")
sim

## 5. Percentual de nulos por coluna

Primeira avaliação de completude das colunas.

**Limitações:** (1) calculado sobre apenas 100 linhas, servindo como indicativo; (2) nessas bases, o código `9` significa "ignorado" e não é contado como nulo pelo pandas, devendo ser tratado explicitamente no ETL.

In [5]:
for nome, df in [("SINASC", sinasc), ("SIM", sim)]:
    print(f"\n=== {nome} ===")
    print((df.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())


=== SINASC ===
CODANOMAL     100.0
IDADEPAI       95.0
DTULTMENST     33.0
SERIESCMAE     31.0
CODMUNNATU      3.0
CODUFNATU       3.0
NATURALMAE      3.0
ESCMAE          0.0
ESTCIVMAE       0.0
contador        0.0
CODESTAB        0.0
CODMUNNASC      0.0
LOCNASC         0.0
ORIGEM          0.0
GRAVIDEZ        0.0
PARTO           0.0
DTNASC          0.0
CONSULTAS       0.0
SEXO            0.0
APGAR1          0.0
APGAR5          0.0
HORANASC        0.0
RACACOR         0.0
PESO            0.0
CODOCUPMAE      0.0
QTDFILVIVO      0.0
QTDFILMORT      0.0
CODMUNRES       0.0
GESTACAO        0.0
IDADEMAE        0.0
DIFDATA         0.0
DTRECEBIM       0.0
VERSAOSIST      0.0
NUMEROLOTE      0.0
DTCADASTRO      0.0
IDANOMAL        0.0
OPORT_DN        0.0
DTRECORIGA      0.0
DTNASCMAE       0.0
ESCMAE2010      0.0
RACACORMAE      0.0
QTDGESTANT      0.0
QTDPARTCES      0.0
QTDPARTNOR      0.0
SEMAGESTAC      0.0
TPMETESTIM      0.0
CONSPRENAT      0.0
MESPRENAT       0.0
TPAPRESENT      0.0
STTR

## 6. Filtro dos óbitos maternos no SIM

Lê o SIM completo em blocos e mantém apenas óbitos femininos com causa básica no capítulo XV da CID-10 (códigos O00–O99) ou com o campo `CAUSAMAT` preenchido.

**Descoberta:** o `CAUSAMAT` contém o código O93, uma adaptação brasileira que marca óbitos maternos associados a causas externas (ex.: O93.0 = acidente de transporte). Segundo o protocolo de codificações especiais do Ministério da Saúde, esses casos não entram no cálculo oficial da Razão de Mortalidade Materna (RMM).

In [ ]:
blocos = []
for bloco in pd.read_csv(ARQUIVOS["SIM"], sep=";", encoding="latin-1",
                         dtype=str, chunksize=200_000):
    materno = bloco[
        (bloco["SEXO"] == "2") &
        (bloco["CAUSABAS"].str.startswith("O", na=False) |
         bloco["CAUSAMAT"].notna())
    ]
    blocos.append(materno)

sim_materno = pd.concat(blocos, ignore_index=True)
print(f"Óbitos maternos em 2024: {len(sim_materno):,}".replace(",", "."))

# quantos vieram só pelo CAUSAMAT (causa básica fora do capítulo O)
so_causamat = ~sim_materno["CAUSABAS"].str.startswith("O", na=False)
print(f"Identificados só pelo CAUSAMAT: {so_causamat.sum()}")

sim_materno.head(100)

## 7. Validação da regra de filtro

Cruza a classificação `MAT_CLAS` com o capítulo da causa básica e conta as mortes maternas tardias (O96/O97), que ocorrem após 42 dias do parto e também ficam fora da RMM.
**Resultado:** 1.610 óbitos filtrados: 1.326 que entram no cálculo da RMM, 258 maternos tardios (O96/O97) e 26 associados a causas externas (marcador O93), incluindo homicídios (O93.7) e suicídios (O93.6).


In [ ]:
print(f"Total filtrado: {len(sim_materno)}")

print("\n=== MAT_CLAS ===")
print(sim_materno["MAT_CLAS"].value_counts(dropna=False))

print("\n=== Letra da causa básica x MAT_CLAS ===")
print(pd.crosstab(sim_materno["CAUSABAS"].str[0], sim_materno["MAT_CLAS"]))

print("\n=== Mortes tardias (O96/O97) ===")
print(sim_materno["CAUSABAS"].str[:3].isin(["O96", "O97"]).sum())

print("\n=== Casos que entraram só pelo CAUSAMAT ===")
print(sim_materno.loc[so_causamat, ["CAUSABAS", "CAUSAMAT", "MAT_CLAS"]]
      .value_counts().head(20))

## 8. Validação das causas com a CID-10 oficial

Cruza o `CAUSABAS` com a tabela oficial de subcategorias da CID-10 do DATASUS para obter a descrição de cada causa e verificar a qualidade dos códigos.

**Regras de veracidade aplicadas:**
- códigos sem correspondência na tabela oficial;
- códigos marcados como `CAUSAOBITO = N`, que não podem ser causa básica de óbito (resultado: 0).

**Destaque:** a causa mais frequente é o O96 (morte materna tardia), reforçando a necessidade de separá-la do cálculo da RMM.

In [ ]:
cid = pd.read_csv(BRUTOS / "cid10" / "CID-10-SUBCATEGORIAS.CSV",
                  sep=";", encoding="latin-1", dtype=str)
cid = cid.loc[:, ~cid.columns.str.startswith("Unnamed")] 
print(f"{len(cid)} códigos CID-10")

sim_cid = sim_materno.merge(cid[["SUBCAT", "DESCRICAO", "CAUSAOBITO"]],
                            left_on="CAUSABAS", right_on="SUBCAT", how="left")

print("Sem correspondência:", sim_cid["DESCRICAO"].isna().sum())

print("Causa básica inválida:", (sim_cid["CAUSAOBITO"] == "N").sum())

sim_cid[["CAUSABAS", "DESCRICAO"]].value_counts().head(15)

## 9. Investigação dos códigos sem correspondência

Lista os 25 códigos que não foram encontrados na tabela de subcategorias.

**Resultado:**
- 4 óbitos com códigos registrados apenas no nível de categoria (O72, O14, O62);
- 21 óbitos com códigos incluídos em atualizações da CID-10 posteriores ao arquivo do DATASUS (2008): O98.7 e O14.2.

In [ ]:
sim_cid.loc[sim_cid["DESCRICAO"].isna(), "CAUSABAS"].value_counts()

## 10. Descrição das causas em cascata

Atribui a descrição de cada causa em três etapas, nesta ordem:
1. tabela oficial de subcategorias;
2. tabela manual com os códigos de atualizações recentes (fonte: CID-10 da OMS);
3. tabela oficial de categorias (3 primeiros caracteres).

A tabela manual vem antes da categoria para que códigos específicos (ex.: O98.7) não recebam a descrição genérica da categoria.

**Resultado:** 0 códigos sem correspondência, ou seja, 100% das causas básicas validadas.

In [ ]:
cat = pd.read_csv(BRUTOS / "cid10" / "CID-10-CATEGORIAS.CSV",
                  sep=";", encoding="latin-1", dtype=str)

cid_manual = pd.DataFrame({
    "SUBCAT": ["O987", "O142"],
    "DESCRICAO": [
        "Doença pelo HIV complicando a gravidez, o parto e o puerpério",
        "Síndrome HELLP",
    ],
})

desc_sub = sim_materno["CAUSABAS"].map(cid.set_index("SUBCAT")["DESCRICAO"])
desc_cat = sim_materno["CAUSABAS"].str[:3].map(cat.set_index("CAT")["DESCRICAO"])
desc_man = sim_materno["CAUSABAS"].map(cid_manual.set_index("SUBCAT")["DESCRICAO"])

sim_materno["DESCRICAO"] = desc_sub.fillna(desc_man).fillna(desc_cat)
print("Sem correspondência:", sim_materno["DESCRICAO"].isna().sum())

## 11. Classificação dos óbitos maternos

Cria a coluna `tipo_obito` a partir da CID-10, separando os óbitos que entram no cálculo da RMM, os maternos tardios (O96/O97) e os associados a causas externas (marcador O93), e compara com a variável `MAT_CLAS` do SIM.

**Resultado:** coincidência integral. Os 1.326 óbitos que entram na RMM têm `MAT_CLAS = 1`, e os 283 tardios e de causa externa têm `MAT_CLAS = 3`. Única exceção: um O97 com `MAT_CLAS = 0`, cuja morte não ocorreu na gravidez nem no puerpério (`TPMORTEOCO = 8`), mantido fora da RMM.

In [ ]:
import numpy as np

cap_o = sim_materno["CAUSABAS"].str.startswith("O", na=False)
tardia = sim_materno["CAUSABAS"].str[:3].isin(["O96", "O97"])

sim_materno["tipo_obito"] = np.select(
    [cap_o & ~tardia, tardia, ~cap_o],
    ["Materno (entra na RMM)", "Materno tardio", "Associado a causa externa"],
    default="Não classificado",
)

print(pd.crosstab(sim_materno["tipo_obito"], sim_materno["MAT_CLAS"]))
sim_materno.loc[sim_materno["MAT_CLAS"] == "0",
                ["CAUSABAS", "DESCRICAO", "TPMORTEOCO"]]

# 12. Comparação com a classificação oficial do SIM

Conta todos os registros com `MAT_CLAS = 1` no arquivo completo, sem o filtro do projeto, e compara caso a caso com a regra construída.

**Resultado:** 1.326 nos dois lados, diferença zero. O total também é idêntico ao Painel de Monitoramento da Mortalidade Materna do Ministério da Saúde para 2024 (óbitos por residência, indicador "Maternos declarados", consulta em 03/10/2026).

In [ ]:
blocos = []
for bloco in pd.read_csv(ARQUIVOS["SIM"], sep=";", encoding="latin-1",
                         dtype=str, chunksize=200_000):
    blocos.append(bloco[bloco["MAT_CLAS"] == "1"])
mat1 = pd.concat(blocos, ignore_index=True)

nossos = sim_materno.loc[sim_materno["tipo_obito"] == "Materno (entra na RMM)", "contador"]
diferenca = mat1[~mat1["contador"].isin(nossos)]

print(f"MAT_CLAS = 1 no SIM inteiro: {len(mat1)}")
print(f"Nossa regra:                {len(nossos)}")
print(f"Diferença:                  {len(diferenca)}")

# 13. Validação por Unidade da Federação

Compara os óbitos maternos por UF de residência (dois primeiros dígitos do `CODMUNRES`) com o Painel de Monitoramento da Mortalidade Materna.

**Resultado:** as 27 UFs apresentaram diferença zero em relação ao número oficial de 2024.

In [ ]:
nossos_uf = (sim_materno[sim_materno["tipo_obito"] == "Materno (entra na RMM)"]
             ["CODMUNRES"].str[:2].value_counts().sort_index())

oficial = {"11": 7, "12": 7, "13": 41, "14": 16, "15": 82, "16": 9, "17": 16,
           "21": 66, "22": 35, "23": 54, "24": 26, "25": 27, "26": 50, "27": 23,
           "28": 15, "29": 97, "31": 125, "32": 24, "33": 100, "35": 201,
           "41": 82, "42": 44, "43": 53, "50": 17, "51": 27, "52": 67, "53": 15}

comparacao = pd.DataFrame({"projeto": nossos_uf, "oficial": pd.Series(oficial)})
comparacao["diferenca"] = comparacao["projeto"] - comparacao["oficial"]
print("UFs com diferença:", (comparacao["diferenca"] != 0).sum())
comparacao

# 14. Perfil do SINASC 2024

**Volume:** 2.389.325 nascidos vivos, compatível com o denominador oficial (1.347 ÷ 2.389.325 × 100.000 = 56,4, RMM divulgada para 2024). Com os óbitos do projeto, a RMM de 2024 é de 55,5 por 100 mil nascidos vivos.

**Completude das variáveis principais:**
- `RACACORMAE`: 1,9% sem informação (vazio);
- `PARTO`: 0,05% ignorado (código 9 ou vazio); 60,6% dos nascimentos foram cesáreas;
- `TPROBSON`: 100% preenchido; grupo 11 (1,6%) indica não classificado;
- `ESCMAE2010`: 1,1% ignorado, registrado tanto como código 9 quanto como vazio.

**Decisões:**
- código 9 e vazio são unificados como "Ignorado";
- `TPROBSON` convertido para inteiro (1 a 11);
- uso da `ESCMAE2010` (padrão atual da DNV) no lugar da `ESCMAE`, compatível com a `ESC2010` do SIM;
- a agregação gera cerca de 363 mil linhas para 2024, compatível com o limite de 1 GB do banco.

In [ ]:
COLS = ["CODMUNRES", "IDADEMAE", "RACACORMAE", "ESCMAE2010",
        "PARTO", "TPROBSON", "CONSPRENAT"]

contagens = {c: pd.Series(dtype="int64") for c in ["RACACORMAE", "PARTO", "TPROBSON", "ESCMAE2010"]}
total = 0
grupos = []

for bloco in pd.read_csv(ARQUIVOS["SINASC"], sep=";", encoding="latin-1",
                         dtype=str, usecols=COLS, chunksize=500_000):
    total += len(bloco)
    for c in contagens:
        contagens[c] = contagens[c].add(bloco[c].value_counts(dropna=False), fill_value=0)
    # simula a agregação (sem faixa etária ainda, então é uma estimativa por cima)
    grupos.append(bloco.groupby(["CODMUNRES", "RACACORMAE", "ESCMAE2010",
                                 "PARTO", "TPROBSON"], dropna=False).size())

print(f"Nascidos vivos 2024: {total:,}".replace(",", "."))

for c, s in contagens.items():
    print(f"\n=== {c} ===")
    print(s.astype(int).sort_index().to_string())

agregado = pd.concat(grupos).groupby(level=[0, 1, 2, 3, 4], dropna=False).sum()
print(f"\nLinhas após agregação (estimativa): {len(agregado):,}".replace(",", "."))

In [ ]:
for c in ["TPROBSON", "ESCMAE2010"]:
    print(f"\n=== {c} ===")
    print(contagens[c].astype(int).sort_index().to_string())

 ## 15. Conversão do SIM 2022 e 2023 (DBC → CSV)

In [3]:
import csv
import pyreaddbc
from dbfread import DBF

for ano in [2022, 2023]:
    dbc = BRUTOS / f"DOBR{ano}.dbc"
    dbf = BRUTOS / f"DOBR{ano}.dbf"
    saida = BRUTOS / f"sim_{ano}.csv"

    pyreaddbc.dbc2dbf(str(dbc), str(dbf))          # descompacta o DBC
    tabela = DBF(dbf, encoding="latin-1")

    with open(saida, "w", newline="", encoding="latin-1") as f:
        escritor = csv.DictWriter(f, fieldnames=tabela.field_names, delimiter=";")
        escritor.writeheader()
        for registro in tabela:
            escritor.writerow(registro)

    dbf.unlink()                                     # apaga o DBF intermediário
    print(f"SIM {ano}: {len(tabela.field_names)} colunas convertidas")

SIM 2022: 87 colunas convertidas
SIM 2023: 87 colunas convertidas


## 16. Comparação do layout entre os anos (2022 a 2024)

Verifica se as colunas dos arquivos mudam de um ano para o outro, para garantir que o ETL funcione igual nos três anos. Lê apenas o cabeçalho de cada arquivo.

**Resultado:** todas as colunas utilizadas no projeto estão presentes nos três anos, nas duas bases. As diferenças encontradas são:
- caixa do nome da coluna `contador`/`CONTADOR`, resolvida pela padronização dos nomes em maiúsculas no ETL;
- colunas incluídas no SIM apenas em 2024 (`MAT_CLAS`, `COVID_CLAS`, `TP_ALTERA`, `CB_ALT`, `OPOR_DO`), o que reforça a decisão de construir a classificação de óbito materno a partir da CID-10;
- colunas de controle presentes apenas nos arquivos DBC (`ESTABDESCR`, `NUDIASOBIN`, `NUDIASINF`, `FONTESINF`), não utilizadas;
- coluna `OPORT_DN` ausente no SINASC 2022, não utilizada.

In [6]:
ANOS = [2022, 2023, 2024]

for base in ["sim", "sinasc"]:
    colunas = {}
    for ano in ANOS:
        arq = BRUTOS / f"{base}_{ano}.csv"
        colunas[ano] = set(pd.read_csv(arq, sep=";", encoding="latin-1",
                                       nrows=0).columns)

    print(f"\n=== {base.upper()} ===")
    for ano in ANOS:
        print(f"{ano}: {len(colunas[ano])} colunas")

    todas = set.union(*colunas.values())
    diferencas = 0
    for col in sorted(todas):
        faltando = [ano for ano in ANOS if col not in colunas[ano]]
        if faltando:
            print(f"  {col}: ausente em {faltando}")
            diferencas += 1
    if diferencas == 0:
        print("  Mesmas colunas nos três anos.")


=== SIM ===
2022: 87 colunas
2023: 87 colunas
2024: 88 colunas
  CB_ALT: ausente em [2022, 2023]
  CONTADOR: ausente em [2024]
  COVID_CLAS: ausente em [2022, 2023]
  ESTABDESCR: ausente em [2024]
  FONTESINF: ausente em [2024]
  MAT_CLAS: ausente em [2022, 2023]
  NUDIASINF: ausente em [2024]
  NUDIASOBIN: ausente em [2024]
  OPOR_DO: ausente em [2022, 2023]
  TP_ALTERA: ausente em [2022, 2023]
  contador: ausente em [2022, 2023]

=== SINASC ===
2022: 61 colunas
2023: 62 colunas
2024: 62 colunas
  CONTADOR: ausente em [2023, 2024]
  OPORT_DN: ausente em [2022]
  contador: ausente em [2022]


## 17. Óbitos maternos por ano (2022 a 2024)

Aplica a regra de classificação validada em 2024 aos três anos, para comparação com o Painel de Monitoramento da Mortalidade Materna.

In [8]:
resumo = {}
por_uf = {}

for ano in ANOS:
    df = pd.read_csv(BRUTOS / f"sim_{ano}.csv", sep=";", encoding="latin-1", dtype=str,
                     usecols=lambda c: c.upper() in ["SEXO", "CAUSABAS", "CAUSAMAT", "CODMUNRES"])
    df.columns = df.columns.str.upper()
    f = df[df["SEXO"] == "2"]
    cap_o = f["CAUSABAS"].str.startswith("O", na=False)
    tardia = f["CAUSABAS"].str[:3].isin(["O96", "O97"])
    externa = ~cap_o & f["CAUSAMAT"].notna() & (f["CAUSAMAT"] != "")

    resumo[ano] = {"Entra na RMM": (cap_o & ~tardia).sum(),
                   "Tardio": (cap_o & tardia).sum(),
                   "Causa externa": externa.sum()}
    por_uf[ano] = f.loc[cap_o & ~tardia, "CODMUNRES"].str[:2].value_counts().sort_index()

print(pd.DataFrame(resumo))
pd.DataFrame(por_uf)

               2022  2023  2024
Entra na RMM   1370  1324  1326
Tardio          270   280   258
Causa externa    35    25    26


,2022,2023,2024
CODMUNRES,,,
11,11,9,7
12,5,6,7
13,53,62,41
14,19,16,16
15,99,83,82
16,11,12,9
17,20,14,16
21,76,84,66
22,37,23,35


## 18. Perfil do SINASC por ano (2022 a 2024)

Repete o perfil da seção 14 para os três anos, lado a lado, para verificar se os códigos e a completude das variáveis usadas no ETL se mantêm entre os anos.

**Resultado:** os códigos das variáveis utilizadas são os mesmos nos três anos, e a proporção de campos vazios diminui de 2022 para 2024 em todas elas. Destaques:
- taxa de cesárea (sobre partos com tipo informado): 58,1% (2022), 59,6% (2023) e 60,6% (2024);
- `TPROBSON` 100% preenchido nos três anos;
- em `STCESPARTO`, o código 3 ("não se aplica") coincide com o número de partos vaginais, confirmando a interpretação do campo; a proporção de cesáreas realizadas antes do trabalho de parto subiu de 52% (2022) para 55% (2024);
- `CONSPRENAT` e `IDADEMAE` usam 99 como código de ignorado, tratado no ETL;
- há registros com idade da mãe inferior a 10 anos, mantidos na faixa "<10".

In [10]:
COLS_SINASC = ["RACACORMAE", "ESCMAE2010", "PARTO", "TPROBSON",
               "STCESPARTO", "STTRABPART", "CONSPRENAT", "IDADEMAE"]

totais = {}
contagens = {c: {} for c in COLS_SINASC[:6]}
consultas = {}
idades = {}

for ano in ANOS:
    total = 0
    parciais = {c: pd.Series(dtype="int64") for c in contagens}
    cons, idd = [], []

    for bloco in pd.read_csv(BRUTOS / f"sinasc_{ano}.csv", sep=";", encoding="latin-1",
                             dtype=str, keep_default_na=False, chunksize=500_000,
                             usecols=lambda c: c.upper() in COLS_SINASC):
        bloco.columns = bloco.columns.str.upper()
        total += len(bloco)
        for c in parciais:
            parciais[c] = parciais[c].add(bloco[c].value_counts(), fill_value=0)
        cons.append(pd.to_numeric(bloco["CONSPRENAT"], errors="coerce"))
        idd.append(pd.to_numeric(bloco["IDADEMAE"], errors="coerce"))

    totais[ano] = total
    for c in contagens:
        contagens[c][ano] = parciais[c].astype(int)
    consultas[ano] = pd.concat(cons).describe()
    idades[ano] = pd.concat(idd).describe()

print("=== Nascidos vivos ===")
print(pd.Series(totais).map("{:,}".format).str.replace(",", "."))

for c, por_ano in contagens.items():
    print(f"\n=== {c} ===")
    print(pd.DataFrame(por_ano).fillna(0).astype(int).sort_index().to_string())

print("\n=== CONSPRENAT (número de consultas) ===")
print(pd.DataFrame(consultas).round(1).to_string())
print("\n=== IDADEMAE ===")
print(pd.DataFrame(idades).round(1).to_string())

=== Nascidos vivos ===
2022    2.561.922
2023    2.537.576
2024    2.389.325
dtype: object

=== RACACORMAE ===
               2022     2023     2024
RACACORMAE                           
              66506    52509    45610
1            849216   837628   786165
2            183434   196378   189894
3             11710    11934    10904
4           1423965  1409599  1327852
5             27091    29528    28900

=== ESCMAE2010 ===
               2022     2023     2024
ESCMAE2010                           
              26687    20077    20673
0              8211     7374     6143
1             63742    56743    47816
2            482132   455983   404318
3           1378332  1389009  1318118
4            129850   127579   121855
5            461903   473655   464662
9             11065     7156     5740

=== PARTO ===
          2022     2023     2024
PARTO                           
          1139      933     1132
1      1072287  1024561   940461
2      1488423  1512021  1447691
9    

## 19. Validação das tabelas tratadas

Calcula, a partir dos arquivos de `dados/tratados`, os indicadores que serão comparados com os painéis oficiais: total de nascidos vivos por ano, RMM e percentual de cesáreas.

**Resultado:** os indicadores calculados a partir das tabelas tratadas coincidem com os obtidos diretamente dos dados brutos, confirmando que o tratamento e a agregação não alteraram nenhuma contagem.

**Validação externa:** o total de nascidos vivos coincide exatamente com o TabNet/DATASUS (Nascidos Vivos – Brasil, por residência da mãe) nos três anos: 2.561.922 (2022), 2.537.576 (2023) e 2.389.325 (2024). Consulta em 03/10/2026.

In [11]:
TRATADOS = RAIZ / "dados" / "tratados"

obitos = pd.read_csv(TRATADOS / "fato_obito_materno.csv", dtype={"cod_mun_res": str})
nasc_mun = pd.read_csv(TRATADOS / "fato_nascidos_municipio.csv", dtype={"cod_mun_res": str})
nasc_parto = pd.read_csv(TRATADOS / "fato_nascidos_parto.csv")

# Nascidos vivos e RMM por ano
nascidos = nasc_mun.groupby("ano")["qtd_nascidos"].sum()
obitos_rmm = obitos[obitos["entra_rmm"]].groupby("ano").size()

resumo = pd.DataFrame({"nascidos_vivos": nascidos, "obitos_rmm": obitos_rmm})
resumo["rmm"] = (resumo["obitos_rmm"] / resumo["nascidos_vivos"] * 100_000).round(1)

# Percentual de cesáreas (código 2), sobre os partos com tipo informado
cesareas = nasc_parto[nasc_parto["cod_parto"] == 2].groupby("ano")["qtd_nascidos"].sum()
informados = nasc_parto[nasc_parto["cod_parto"].isin([1, 2])].groupby("ano")["qtd_nascidos"].sum()
resumo["pct_cesarea"] = (cesareas / informados * 100).round(1)

resumo

,nascidos_vivos,obitos_rmm,rmm,pct_cesarea
ano,,,,
2022,2561922,1370,53.5,58.1
2023,2537576,1324,52.2,59.6
2024,2389325,1326,55.5,60.6
